## 1. Initialize Project Environment
Import libraries for phylogenetic tree construction and analysis.

In [1]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List

import pandas as pd
from Bio import Phylo, SeqIO
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

## 2. Define Configuration Parameters
Centralize paths and tree construction options.

In [2]:
@dataclass
class TreeConfig:
    handle: str
    fasta_path: Path = None
    export_dir: Path = Path("artifacts")
    distance_model: str = "identity"

    def __post_init__(self):
        if self.fasta_path is None:
            self.fasta_path = Path(
                f"../../../data/work/{self.handle}/lab04/tp53_multi_sequences.fasta"
            )

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["fasta_path"] = str(info["fasta_path"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = TreeConfig(handle="AndreiCod")
CONFIG.describe()

{'handle': 'AndreiCod',
 'fasta_path': '../../../data/work/AndreiCod/lab04/tp53_multi_sequences.fasta',
 'export_dir': 'artifacts',
 'distance_model': 'identity'}

In [3]:
def load_sequences(cfg: TreeConfig) -> List[SeqRecord]:
    """Load sequences from FASTA file."""
    if not cfg.fasta_path.exists():
        raise FileNotFoundError(f"FASTA not found: {cfg.fasta_path}. Run Task1 first.")

    records = list(SeqIO.parse(cfg.fasta_path, "fasta"))
    logging.info("Loaded %d sequences from %s", len(records), cfg.fasta_path)
    return records


sequences = load_sequences(CONFIG)
print(f"Loaded {len(sequences)} sequences:")
for rec in sequences:
    print(f"  {rec.id}: {len(rec.seq)} bp")

[INFO] Loaded 10 sequences from ../../../data/work/AndreiCod/lab04/tp53_multi_sequences.fasta


Loaded 10 sequences:
  NM_000546.6: 2512 bp
  NM_011640.3: 1781 bp
  NM_131327.2: 2233 bp
  XM_006719566.3: 3347 bp
  NM_001317019.1: 4747 bp
  NM_001085860.1: 1719 bp
  XM_005194938.2: 5035 bp
  NM_001006919.1: 1515 bp
  NM_001089263.1: 2415 bp
  XM_031279688.1: 808 bp


## 3. Implement Core Functionality
Create alignment, compute distances, and build Neighbor-Joining tree.

In [4]:
def create_padded_alignment(records: List[SeqRecord]) -> MultipleSeqAlignment:
    """Create alignment by padding sequences to equal length."""
    max_len = max(len(rec.seq) for rec in records)
    aligned = []

    for rec in records:
        padded_seq = str(rec.seq) + "-" * (max_len - len(rec.seq))
        aligned.append(SeqRecord(Seq(padded_seq), id=rec.id, description=""))

    return MultipleSeqAlignment(aligned)


alignment = create_padded_alignment(sequences)
logging.info(
    "Created alignment: %d sequences × %d bp",
    len(alignment),
    alignment.get_alignment_length(),
)

[INFO] Created alignment: 10 sequences × 5035 bp


In [5]:
def build_nj_tree(alignment: MultipleSeqAlignment, model: str = "identity"):
    """Build Neighbor-Joining tree from alignment."""
    calculator = DistanceCalculator(model)
    distance_matrix = calculator.get_distance(alignment)

    constructor = DistanceTreeConstructor()
    tree = constructor.nj(distance_matrix)

    return tree, distance_matrix


tree, distance_matrix = build_nj_tree(alignment, CONFIG.distance_model)

print("Distance Matrix:")
print(distance_matrix)
print("\nNeighbor-Joining Tree:")
Phylo.draw_ascii(tree)

Distance Matrix:
NM_000546.6 0.000000
NM_011640.3 0.400596    0.000000
NM_131327.2 0.389275    0.346574    0.000000
XM_006719566.3  0.544786    0.575571    0.549553    0.000000
NM_001317019.1  0.820854    0.848659    0.830189    0.767627    0.000000
NM_001085860.1  0.414300    0.269315    0.359881    0.582125    0.857795    0.000000
XM_005194938.2  0.876663    0.909037    0.890566    0.832175    0.748957    0.912214    0.000000
NM_001006919.1  0.426018    0.283615    0.368620    0.585501    0.871698    0.270904    0.928898    0.000000
NM_001089263.1  0.371797    0.396624    0.370804    0.534657    0.820060    0.400000    0.885799    0.396226    0.000000
XM_031279688.1  0.454618    0.312413    0.402582    0.625819    0.902880    0.305462    0.961470    0.259583    0.441907    0.000000
    NM_000546.6 NM_011640.3 NM_131327.2 XM_006719566.3  NM_001317019.1  NM_001085860.1  XM_005194938.2  NM_001006919.1  NM_001089263.1  XM_031279688.1

Neighbor-Joining Tree:
                           ___

## 4. Analyze Tree Clusters
Extract cluster statistics for notes.md (insights documented there, not in code).

In [6]:
def extract_tree_stats(tree) -> Dict:
    """Extract quantitative tree statistics for notes.md."""
    terminals = list(tree.get_terminals())
    internals = list(tree.get_nonterminals())

    # Get branch lengths
    branch_lengths = [c.branch_length for c in tree.find_clades() if c.branch_length]

    stats = {
        "num_terminals": len(terminals),
        "num_internal_nodes": len(internals),
        "total_tree_length": round(sum(branch_lengths), 4) if branch_lengths else None,
        "mean_branch_length": round(sum(branch_lengths) / len(branch_lengths), 4)
        if branch_lengths
        else None,
        "min_branch_length": round(min(branch_lengths), 4) if branch_lengths else None,
        "max_branch_length": round(max(branch_lengths), 4) if branch_lengths else None,
        "terminal_names": [t.name for t in terminals],
    }

    return stats


tree_stats = extract_tree_stats(tree)

print("Tree Statistics:")
for k, v in tree_stats.items():
    if k != "terminal_names":
        print(f"  {k}: {v}")
print(f"\nTerminals: {len(tree_stats['terminal_names'])} species")

Tree Statistics:
  num_terminals: 10
  num_internal_nodes: 8
  total_tree_length: 2.4589
  mean_branch_length: 0.1446
  min_branch_length: 0.0052
  max_branch_length: 0.4043

Terminals: 10 species


## 5. Export Results
Save tree in Newick format and interpretation to artifacts.

In [7]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save tree in Newick format
tree_path = EXPORT_DIR / "task2_nj_tree.nwk"
Phylo.write(tree, tree_path, "newick")
print(f"[OK] NJ tree saved to: {tree_path.resolve()}")

# Save tree stats as JSON (for reproducibility)
import json

stats_path = EXPORT_DIR / "task2_tree_stats.json"
with open(stats_path, "w") as f:
    json.dump(tree_stats, f, indent=2)
print(f"[OK] Tree statistics saved to: {stats_path.resolve()}")

print(f"\nArtifacts saved to {EXPORT_DIR.resolve()}")

[OK] NJ tree saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task2_nj_tree.nwk
[OK] Tree statistics saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task2_tree_stats.json

Artifacts saved to /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts
